# §0.4 カルマンフィルタとベイズ推定 - 情報幾何への接続

## 1. 概要

- **この節で学ぶこと**: カルマンフィルタの幾何学的解釈、ベイズ更新とFisher情報の関係、情報利得の計算
- **前提知識**: 正規分布、ベイズの定理、条件付き確率
- **情報幾何との関連**: **ベイズ更新は統計多様体上の「移動」**

## 2. 直感的理解

### カルマンフィルタは「最適な情報統合」

- **予測ステップ**: 動力学モデルに基づく状態予測（不確実性が増加）
- **更新ステップ**: 観測に基づくベイズ更新（不確実性が減少）

### 幾何学的視点

- 事前分布 → 事後分布の変化は、**正規分布多様体上の移動**
- 移動距離（Fisher距離）= 得られた情報量
- 精度（分散の逆数）が大きいほど、同じ更新でも「遠く」移動

### EKFとの接続

- 拡張カルマンフィルタ（EKF）は非線形関数の線形化
- 線形化 = **接空間での近似**（微分幾何の概念！）

## 3. 数学的定義

### 3.1 線形ガウス状態空間モデル

**状態方程式**:
$$x_t = A x_{t-1} + w_t, \quad w_t \sim N(0, Q)$$

**観測方程式**:
$$y_t = H x_t + v_t, \quad v_t \sim N(0, R)$$

### 3.2 カルマンフィルタのアルゴリズム

**予測ステップ**:
$$\hat{x}_{t|t-1} = A \hat{x}_{t-1|t-1}$$
$$P_{t|t-1} = A P_{t-1|t-1} A^\top + Q$$

**更新ステップ**:
$$K_t = P_{t|t-1} H^\top (H P_{t|t-1} H^\top + R)^{-1}$$
$$\hat{x}_{t|t} = \hat{x}_{t|t-1} + K_t (y_t - H \hat{x}_{t|t-1})$$
$$P_{t|t} = (I - K_t H) P_{t|t-1}$$

### 3.3 精度行列の更新（情報フィルタ形式）

精度行列 $\Lambda = P^{-1}$ を使うと:
$$\Lambda_{t|t} = \Lambda_{t|t-1} + H^\top R^{-1} H$$

**観測のFisher情報 $H^\top R^{-1} H$ が加算される！**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

def kl_divergence_gaussian(mu1, sigma1, mu2, sigma2):
    """D_KL(N(μ1,σ1²) || N(μ2,σ2²))"""
    return np.log(sigma2/sigma1) + (sigma1**2 + (mu1-mu2)**2)/(2*sigma2**2) - 0.5

def fisher_distance_gaussian(mu1, sigma1, mu2, sigma2):
    """正規分布間のFisher-Rao距離（解析解）"""
    # Fisher-Rao距離の近似（正確な解は複雑）
    return np.sqrt(2) * np.abs(np.log(sigma2/sigma1)) + np.abs(mu1-mu2)/np.sqrt(sigma1*sigma2)

## 4. 可視化

### 4.1 1次元カルマンフィルタの動作

In [ ]:
def kalman_filter_1d(y_obs, A=1.0, H=1.0, Q=0.1, R=1.0, x0=0, P0=1.0):
    """1D Kalman filter implementation"""
    n = len(y_obs)
    
    x_pred = np.zeros(n)
    P_pred = np.zeros(n)
    x_filt = np.zeros(n)
    P_filt = np.zeros(n)
    K = np.zeros(n)
    
    x_prev, P_prev = x0, P0
    
    for t in range(n):
        # Prediction step
        x_pred[t] = A * x_prev
        P_pred[t] = A**2 * P_prev + Q
        
        # Update step
        K[t] = P_pred[t] * H / (H**2 * P_pred[t] + R)
        x_filt[t] = x_pred[t] + K[t] * (y_obs[t] - H * x_pred[t])
        P_filt[t] = (1 - K[t] * H) * P_pred[t]
        
        x_prev, P_prev = x_filt[t], P_filt[t]
    
    return x_pred, P_pred, x_filt, P_filt, K

# Simulation
T = 50
x_true = np.cumsum(np.random.randn(T) * 0.3)  # True state (random walk)
y_obs = x_true + np.random.randn(T) * 1.0      # Observations (with noise)

x_pred, P_pred, x_filt, P_filt, K = kalman_filter_1d(y_obs, Q=0.1, R=1.0)

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

ax1 = axes[0]
ax1.plot(x_true, 'g-', linewidth=2, label='True state')
ax1.plot(y_obs, 'r.', markersize=5, alpha=0.5, label='Observations')
ax1.plot(x_filt, 'b-', linewidth=2, label='Filtered estimate')
ax1.fill_between(range(T), x_filt - 2*np.sqrt(P_filt), x_filt + 2*np.sqrt(P_filt),
                 alpha=0.3, color='blue', label='95% CI')
ax1.set_xlabel('Time t')
ax1.set_ylabel('State x')
ax1.set_title('Kalman Filter State Estimation')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(P_pred, 'r--', label='Predicted variance')
ax2.plot(P_filt, 'b-', label='Filtered variance')
ax2.axhline(y=P_filt[-1], color='gray', linestyle=':', label=f'Steady state {P_filt[-1]:.3f}')
ax2.set_xlabel('Time t')
ax2.set_ylabel('Variance P')
ax2.set_title('Variance Evolution (Prediction increases, Update decreases)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.2 ベイズ更新の幾何学的解釈

In [ ]:
def visualize_bayesian_update_geometry():
    """Visualize Bayesian update as movement on Gaussian manifold"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Prior
    mu_prior, sigma_prior = 0, 2
    
    # Likelihood
    y_obs = 3
    sigma_likelihood = 1
    
    # Posterior (analytical)
    prec_prior = 1 / sigma_prior**2
    prec_likelihood = 1 / sigma_likelihood**2
    prec_post = prec_prior + prec_likelihood
    mu_post = (prec_prior * mu_prior + prec_likelihood * y_obs) / prec_post
    sigma_post = 1 / np.sqrt(prec_post)
    
    # Left: Distribution changes
    ax1 = axes[0]
    x = np.linspace(-5, 7, 200)
    
    prior = stats.norm.pdf(x, mu_prior, sigma_prior)
    likelihood = stats.norm.pdf(x, y_obs, sigma_likelihood)
    posterior = stats.norm.pdf(x, mu_post, sigma_post)
    
    ax1.plot(x, prior, 'b-', linewidth=2, label=f'Prior: N({mu_prior}, {sigma_prior}²)')
    ax1.plot(x, likelihood, 'g--', linewidth=2, label=f'Likelihood: N({y_obs}, {sigma_likelihood}²)')
    ax1.plot(x, posterior, 'r-', linewidth=2, label=f'Posterior: N({mu_post:.2f}, {sigma_post:.2f}²)')
    ax1.axvline(y_obs, color='green', linestyle=':', alpha=0.5)
    ax1.set_xlabel('x')
    ax1.set_ylabel('Density')
    ax1.set_title('Bayesian Update: Prior × Likelihood ∝ Posterior')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Right: Movement in parameter space
    ax2 = axes[1]
    
    # Draw Fisher metric ellipses in background
    theta = np.linspace(0, 2*np.pi, 100)
    for mu_c in np.linspace(-1, 4, 6):
        for sigma_c in [0.5, 1.0, 1.5, 2.0]:
            scale = 0.15
            ellipse_x = sigma_c * np.cos(theta) * scale + mu_c
            ellipse_y = sigma_c / np.sqrt(2) * np.sin(theta) * scale + sigma_c
            ax2.plot(ellipse_x, ellipse_y, color='gray', linewidth=0.5, alpha=0.3)
    
    # Prior → Posterior movement
    ax2.plot(mu_prior, sigma_prior, 'bo', markersize=15, label='Prior')
    ax2.plot(mu_post, sigma_post, 'ro', markersize=15, label='Posterior')
    ax2.annotate('', xy=(mu_post, sigma_post), xytext=(mu_prior, sigma_prior),
                 arrowprops=dict(arrowstyle='->', color='purple', lw=3))
    
    # Information gain
    info_gain = kl_divergence_gaussian(mu_post, sigma_post, mu_prior, sigma_prior)
    
    ax2.set_xlabel('μ')
    ax2.set_ylabel('σ')
    ax2.set_title(f'Movement in Parameter Space\nInfo gain D_KL(post||prior) = {info_gain:.3f} nats')
    ax2.legend()
    ax2.set_xlim(-2, 5)
    ax2.set_ylim(0, 2.5)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Precision change: {1/sigma_prior**2:.3f} → {1/sigma_post**2:.3f}")
    print(f"Precision increase = Observation Fisher info: {1/sigma_likelihood**2:.3f}")

visualize_bayesian_update_geometry()

### 4.3 観測精度と情報利得の関係

In [ ]:
def visualize_observation_precision_effect():
    """Effect of observation precision on information gain"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    mu_prior, sigma_prior = 0, 2
    y_obs = 3
    
    sigma_obs_list = np.linspace(0.3, 3.0, 50)
    info_gains = []
    mu_posts = []
    sigma_posts = []
    
    for sigma_obs in sigma_obs_list:
        prec_prior = 1 / sigma_prior**2
        prec_obs = 1 / sigma_obs**2
        prec_post = prec_prior + prec_obs
        mu_post = (prec_prior * mu_prior + prec_obs * y_obs) / prec_post
        sigma_post = 1 / np.sqrt(prec_post)
        
        info_gain = kl_divergence_gaussian(mu_post, sigma_post, mu_prior, sigma_prior)
        info_gains.append(info_gain)
        mu_posts.append(mu_post)
        sigma_posts.append(sigma_post)
    
    # Left: Info gain vs observation precision
    ax1 = axes[0]
    ax1.plot(1/sigma_obs_list**2, info_gains, 'b-', linewidth=2)
    ax1.set_xlabel('Observation precision 1/σ_obs²')
    ax1.set_ylabel('Information gain (nats)')
    ax1.set_title('Higher Precision → More Information Gain')
    ax1.grid(True, alpha=0.3)
    
    # Right: Trajectory in parameter space
    ax2 = axes[1]
    ax2.plot(mu_prior, sigma_prior, 'bo', markersize=15, label='Prior', zorder=5)
    
    # Show posteriors for different observation precisions
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(sigma_obs_list)))
    ax2.scatter(mu_posts, sigma_posts, c=1/np.array(sigma_obs_list)**2, cmap='viridis', s=30)
    
    # Label some representative points
    for sigma_obs, color in [(0.5, 'red'), (1.0, 'orange'), (2.0, 'green')]:
        idx = np.argmin(np.abs(sigma_obs_list - sigma_obs))
        ax2.plot(mu_posts[idx], sigma_posts[idx], 'o', color=color, markersize=12,
                 label=f'σ_obs={sigma_obs}')
        ax2.annotate('', xy=(mu_posts[idx], sigma_posts[idx]), xytext=(mu_prior, sigma_prior),
                     arrowprops=dict(arrowstyle='->', color=color, lw=1.5, alpha=0.7))
    
    ax2.set_xlabel('μ')
    ax2.set_ylabel('σ')
    ax2.set_title('Posterior Change by Observation Precision\n(Higher precision → closer to observation)')
    ax2.legend()
    ax2.set_xlim(-0.5, 3.5)
    ax2.set_ylim(0, 2.5)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_observation_precision_effect()

### 4.4 2次元カルマンフィルタ（位置・速度）

In [ ]:
def kalman_filter_2d(y_obs, dt=1.0, q=0.1, r=1.0):
    """2D Kalman filter (position and velocity estimation)"""
    # State transition matrix (constant velocity model)
    A = np.array([[1, dt], [0, 1]])
    
    # Observation matrix (position only)
    H = np.array([[1, 0]])
    
    # Process noise covariance
    Q = q * np.array([[dt**3/3, dt**2/2], [dt**2/2, dt]])
    
    # Observation noise covariance
    R = np.array([[r]])
    
    n = len(y_obs)
    x_filt = np.zeros((n, 2))
    P_filt = np.zeros((n, 2, 2))
    
    x = np.array([0, 0])  # Initial state
    P = np.eye(2) * 10    # Initial covariance
    
    for t in range(n):
        # Prediction
        x_pred = A @ x
        P_pred = A @ P @ A.T + Q
        
        # Update
        S = H @ P_pred @ H.T + R
        K = P_pred @ H.T @ np.linalg.inv(S)
        x = x_pred + K.flatten() * (y_obs[t] - H @ x_pred)
        P = (np.eye(2) - K @ H) @ P_pred
        
        x_filt[t] = x
        P_filt[t] = P
    
    return x_filt, P_filt

# Simulation
T = 100
dt = 0.1
t = np.arange(T) * dt

# True trajectory (accelerated motion)
pos_true = 0.5 * t**2  # Constant acceleration
vel_true = t            # Velocity

# Observations (with noise)
y_obs = pos_true + np.random.randn(T) * 0.5

x_filt, P_filt = kalman_filter_2d(y_obs, dt=dt, q=1.0, r=0.25)

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

ax1 = axes[0]
ax1.plot(t, pos_true, 'g-', linewidth=2, label='True position')
ax1.plot(t, y_obs, 'r.', markersize=3, alpha=0.5, label='Observations')
ax1.plot(t, x_filt[:, 0], 'b-', linewidth=2, label='Estimated position')
ax1.fill_between(t, x_filt[:, 0] - 2*np.sqrt(P_filt[:, 0, 0]),
                 x_filt[:, 0] + 2*np.sqrt(P_filt[:, 0, 0]),
                 alpha=0.3, color='blue')
ax1.set_xlabel('Time t')
ax1.set_ylabel('Position x')
ax1.set_title('Position Estimation')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(t, vel_true, 'g-', linewidth=2, label='True velocity')
ax2.plot(t, x_filt[:, 1], 'b-', linewidth=2, label='Estimated velocity')
ax2.fill_between(t, x_filt[:, 1] - 2*np.sqrt(P_filt[:, 1, 1]),
                 x_filt[:, 1] + 2*np.sqrt(P_filt[:, 1, 1]),
                 alpha=0.3, color='blue')
ax2.set_xlabel('Time t')
ax2.set_ylabel('Velocity v')
ax2.set_title('Velocity Estimation (Not directly observed, yet estimated!)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 具体例

### 例1：情報フィルタ形式での更新

In [ ]:
def demonstrate_information_filter():
    """情報フィルタ形式：精度行列の加法性"""
    print("【情報フィルタ形式】")
    print("\n精度行列 Λ = P⁻¹ を使うと、更新式が簡単になる:")
    print("  Λ_post = Λ_prior + Λ_obs")
    print("  η_post = η_prior + η_obs")
    print("  （η = Λμ は自然パラメータ）")
    
    # 事前分布
    mu_prior, sigma_prior = 0, 2
    Lambda_prior = 1 / sigma_prior**2
    eta_prior = Lambda_prior * mu_prior
    
    # 複数の観測
    observations = [(2.0, 1.0), (3.0, 0.5), (2.5, 1.5)]  # (観測値, 観測ノイズσ)
    
    Lambda_curr = Lambda_prior
    eta_curr = eta_prior
    
    print(f"\n初期: μ = {mu_prior}, σ = {sigma_prior}, Λ = {Lambda_prior:.4f}")
    
    for i, (y, sigma_obs) in enumerate(observations):
        Lambda_obs = 1 / sigma_obs**2
        eta_obs = Lambda_obs * y
        
        Lambda_curr = Lambda_curr + Lambda_obs  # 精度の加算！
        eta_curr = eta_curr + eta_obs
        
        mu_curr = eta_curr / Lambda_curr
        sigma_curr = 1 / np.sqrt(Lambda_curr)
        
        print(f"\n観測{i+1}: y={y}, σ_obs={sigma_obs}")
        print(f"  → Λ = {Lambda_curr:.4f}, μ = {mu_curr:.4f}, σ = {sigma_curr:.4f}")
    
    print("\n【ポイント】")
    print("- 精度（Fisher情報）は観測ごとに加算される")
    print("- これは情報幾何における『情報の加法性』の表れ")

demonstrate_information_filter()

### 例2：センサー融合（異なる精度のセンサー）

In [ ]:
def visualize_sensor_fusion():
    """Sensor fusion with different precisions"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # True value
    x_true = 5.0
    
    # Sensor 1: High precision but biased
    y1, sigma1 = 4.8, 0.3
    
    # Sensor 2: Low precision but less biased
    y2, sigma2 = 5.2, 1.0
    
    # Fusion result
    Lambda1 = 1 / sigma1**2
    Lambda2 = 1 / sigma2**2
    Lambda_fused = Lambda1 + Lambda2
    mu_fused = (Lambda1 * y1 + Lambda2 * y2) / Lambda_fused
    sigma_fused = 1 / np.sqrt(Lambda_fused)
    
    # Left: Distribution comparison
    ax1 = axes[0]
    x = np.linspace(3, 7, 200)
    
    ax1.plot(x, stats.norm.pdf(x, y1, sigma1), 'b-', linewidth=2, 
             label=f'Sensor 1: N({y1}, {sigma1}²)')
    ax1.plot(x, stats.norm.pdf(x, y2, sigma2), 'g-', linewidth=2,
             label=f'Sensor 2: N({y2}, {sigma2}²)')
    ax1.plot(x, stats.norm.pdf(x, mu_fused, sigma_fused), 'r-', linewidth=2,
             label=f'Fused: N({mu_fused:.2f}, {sigma_fused:.2f}²)')
    ax1.axvline(x_true, color='black', linestyle='--', label=f'True value: {x_true}')
    
    ax1.set_xlabel('x')
    ax1.set_ylabel('Density')
    ax1.set_title('Sensor Fusion\n(Weighted toward high-precision sensor)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Right: Weight visualization
    ax2 = axes[1]
    weights = [Lambda1/Lambda_fused, Lambda2/Lambda_fused]
    bars = ax2.bar(['Sensor 1\n(High prec.)', 'Sensor 2\n(Low prec.)'], weights, 
                   color=['blue', 'green'])
    
    for bar, w in zip(bars, weights):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{w:.1%}', ha='center', fontsize=14)
    
    ax2.set_ylabel('Weight')
    ax2.set_title('Fusion Weights\n(Proportional to precision)')
    ax2.set_ylim(0, 1.1)
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Sensor 1 precision: {Lambda1:.2f}")
    print(f"Sensor 2 precision: {Lambda2:.2f}")
    print(f"Fused precision: {Lambda_fused:.2f}")
    print(f"\nPrecision additivity: {Lambda1:.2f} + {Lambda2:.2f} = {Lambda_fused:.2f}")

visualize_sensor_fusion()

### 例3：EKFの線形化と接空間

In [ ]:
def demonstrate_ekf_linearization():
    """EKF linearization from tangent space perspective"""
    print("【EKF and Tangent Space】")
    print("\nNonlinear observation model: y = h(x) + v")
    print("Linearization: h(x) ≈ h(x̂) + H(x - x̂)")
    print("where H = ∂h/∂x|_{x=x̂}")
    print("\nGeometric interpretation:")
    print("- h(x) maps points on one manifold to another")
    print("- H is the Jacobian = tangent map")
    print("- Linearization = approximation in tangent space")
    
    # Example: Polar coordinate observation
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Estimated position
    x_hat = np.array([3, 4])  # (x, y)
    
    # Nonlinear observation model: h(x) = [r, θ] = [√(x²+y²), atan2(y,x)]
    r_hat = np.sqrt(x_hat[0]**2 + x_hat[1]**2)
    theta_hat = np.arctan2(x_hat[1], x_hat[0])
    
    # Jacobian
    H = np.array([
        [x_hat[0]/r_hat, x_hat[1]/r_hat],
        [-x_hat[1]/r_hat**2, x_hat[0]/r_hat**2]
    ])
    
    # Plot estimated position
    ax.plot(x_hat[0], x_hat[1], 'ro', markersize=15, label='Estimate x̂')
    
    # Range direction (∂h/∂r)
    dr = x_hat / r_hat
    ax.arrow(x_hat[0], x_hat[1], dr[0]*1.5, dr[1]*1.5, head_width=0.2, 
             head_length=0.1, fc='blue', ec='blue')
    ax.text(x_hat[0] + dr[0]*1.8, x_hat[1] + dr[1]*1.8, '∂r/∂x', fontsize=12, color='blue')
    
    # Angle direction (∂h/∂θ)
    dtheta = np.array([-x_hat[1], x_hat[0]]) / r_hat
    ax.arrow(x_hat[0], x_hat[1], dtheta[0]*1.5, dtheta[1]*1.5, head_width=0.2,
             head_length=0.1, fc='green', ec='green')
    ax.text(x_hat[0] + dtheta[0]*1.8, x_hat[1] + dtheta[1]*1.8, '∂θ/∂x', fontsize=12, color='green')
    
    # Constant range circles
    theta = np.linspace(0, 2*np.pi, 100)
    for r in [3, 4, 5, 6]:
        ax.plot(r*np.cos(theta), r*np.sin(theta), 'k--', alpha=0.3)
    
    # Constant angle lines
    for t in np.linspace(0, np.pi/2, 5):
        ax.plot([0, 7*np.cos(t)], [0, 7*np.sin(t)], 'k--', alpha=0.3)
    
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title('EKF Linearization\nTangent vectors (Jacobian columns) approximation')
    ax.set_xlim(-1, 7)
    ax.set_ylim(-1, 7)
    ax.set_aspect('equal')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.show()
    
    print(f"\nEstimated position: x̂ = {x_hat}")
    print(f"Polar coordinates: (r, θ) = ({r_hat:.2f}, {np.degrees(theta_hat):.1f}°)")
    print(f"\nJacobian H = ")
    print(H)

demonstrate_ekf_linearization()

## 6. 他の概念との関係

### 前の節との繋がり
- **確率・統計 (§0.2)**: 正規分布の共役性がカルマンフィルタの解析解を可能に
- **KL・Fisher (§0.3)**: 観測のFisher情報が精度として加算される

### 情報幾何との接続

| カルマンフィルタ | 情報幾何 |
|:---|:---|
| 精度行列 $\Lambda = P^{-1}$ | Fisher情報行列 $I(\theta)$ |
| 自然パラメータ $\eta = \Lambda\mu$ | 指数型分布族の自然パラメータ |
| ベイズ更新 | 統計多様体上の移動 |
| EKFの線形化 | 接空間での近似 |
| 情報利得 $D_{\text{KL}}(\text{post}\|\text{prior})$ | Fisher距離（の近似） |

### 次の節への接続
- **多様体 (§1.1)**: パラメータ空間が多様体構造を持つ
- **接ベクトル (§1.2)**: EKFのヤコビアンは接写像
- **リーマン計量 (§1.5)**: Fisher情報行列がリーマン計量を定義

## 7. 演習問題

### Q1. 精度の加法性

事前分布 $N(0, 4)$ に対して、精度 $1$ の観測を2回（観測値 $y_1=1$, $y_2=3$）行ったときの事後分布を計算せよ。

<details>
<summary>解答を見る</summary>

事前: $\Lambda_0 = 1/4$, $\eta_0 = 0$

観測1後: $\Lambda_1 = 1/4 + 1 = 5/4$, $\eta_1 = 0 + 1 = 1$
→ $\mu_1 = 4/5 = 0.8$, $\sigma_1 = \sqrt{4/5} \approx 0.894$

観測2後: $\Lambda_2 = 5/4 + 1 = 9/4$, $\eta_2 = 1 + 3 = 4$
→ $\mu_2 = 16/9 \approx 1.78$, $\sigma_2 = \sqrt{4/9} = 2/3 \approx 0.667$

</details>

In [ ]:
# Q1検証
Lambda = 1/4  # 事前精度
eta = 0       # 事前自然パラメータ

for y in [1, 3]:
    Lambda += 1  # 精度1の観測
    eta += y
    mu = eta / Lambda
    sigma = 1 / np.sqrt(Lambda)
    print(f"y={y}観測後: μ={mu:.4f}, σ={sigma:.4f}")

### Q2. 情報利得の計算

上の問題で、2回の観測による総情報利得 $D_{\text{KL}}(\text{final posterior} \| \text{prior})$ を計算せよ。

In [ ]:
# Q2検証
mu_prior, sigma_prior = 0, 2
mu_post, sigma_post = 16/9, 2/3

info_gain = kl_divergence_gaussian(mu_post, sigma_post, mu_prior, sigma_prior)
print(f"総情報利得: {info_gain:.4f} nats")

### Q3. カルマンゲインの解釈

カルマンゲイン $K = P_{\text{pred}} H^\top (H P_{\text{pred}} H^\top + R)^{-1}$ が、観測ノイズ $R \to 0$ のとき $K \to H^{-1}$ となることを示し、その意味を説明せよ。

<details>
<summary>解答を見る</summary>

1次元で $H=1$ の場合:
$$K = \frac{P_{\text{pred}}}{P_{\text{pred}} + R}$$

$R \to 0$ のとき $K \to 1$。

更新式: $\hat{x} = \hat{x}_{\text{pred}} + K(y - \hat{x}_{\text{pred}}) \to y$

**意味**: 観測が完全に信頼できる（ノイズゼロ）なら、推定値は観測値そのものになる。

</details>

## 8. 参考：使用したプロンプト

```
カルマンフィルタの更新式を情報フィルタ形式で書き直してください。
精度行列（Fisher情報）がどのように加算されるか示してください。
```

```
ベイズ更新を正規分布多様体上の「移動」として可視化するPythonコードを書いてください。
パラメータ空間(μ, σ)上で事前→事後の変化を矢印で示してください。
```

```
EKFの線形化を微分幾何の接空間の観点から説明してください。
ヤコビアンが接写像（tangent map）であることを図示してください。
```

```
異なる精度のセンサーを融合するとき、なぜ高精度センサーに
重みが偏るのかを、情報幾何の観点から説明してください。
```

---
**次のステップ**: 第1章 `ch01_differential_geometry/01_manifolds.ipynb` へ